# 15. Vector Databases

**Tier:** Building with LLMs
**Estimated time:** 35 minutes
**Prerequisites:** 03, 14
**Source material:** @sairahul1 — "20 AI Concepts You Must Understand in 2026" (https://x.com/sairahul1/status/2057740928908161461)

## What You'll Learn
- Why brute-force cosine similarity (notebook 14) stops scaling once you have millions of chunks
- What an Approximate Nearest Neighbor (ANN) index trades away — exactness — to buy back speed
- How `ragkit.vectorstore` (Chroma) already gives you this for free, and when Pinecone enters the picture

## Why This Matters
Notebook 14's retrieval step compared the query against every chunk, one at a time. That's fine for ~100 chunks and a death sentence for 10 million. A vector database exists to make retrieval fast at scale without you reimplementing the index yourself.


In [ ]:
import os

# Load .env if python-dotenv is installed; harmless if it isn't.
# (ragkit.config already does this on import, but we load it explicitly here too,
# matching every other notebook, since this one needs PINECONE_API_KEY for the optional cell below.)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

## What changes at scale

Notebook 14's retrieval was brute-force: compute cosine similarity against *every* chunk, then sort. For a few hundred chunks that's instant. For a few million, comparing against every single one, every single query, becomes the bottleneck — both in time and in how much fits in memory.

A **vector database** solves this with an **Approximate Nearest Neighbor (ANN)** index: a data structure (commonly a graph like HNSW, or clustering like IVF) built once over your vectors that lets a query find *probably* the closest matches by visiting a small fraction of the data, not all of it. The "approximate" part is the trade: you give up a small, usually negligible amount of **recall** (chance of finding the true best match) in exchange for retrieval that doesn't scale linearly with corpus size. Vector databases also handle the parts brute force ignores: persistence to disk, metadata filtering (e.g. "only search the `spec` category"), and incremental updates as documents change.


In [ ]:
# Native-library guards: torch, faiss, and chromadb each link their own OpenMP runtime.
# Setting these BEFORE importing them avoids duplicate-runtime segfaults on macOS.
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
sys.path.insert(0, "..")
import time
import numpy as np
import faiss
faiss.omp_set_num_threads(1)
print("faiss version:", faiss.__version__)


## Real retrieval on the Helios corpus (small + exact)

First, the honest version: embed the real corpus (a few dozen chunks) and do exact brute-force search. At this size, brute force is the right tool — no index needed.


In [ ]:
from ragkit.data import load_corpus, chunk_text
from ragkit.embeddings import embed, cosine_similarity

docs = load_corpus()
chunks = []
for doc in docs:
    chunks += chunk_text(doc["text"], chunk_size=120, overlap=20)
print(f"{len(chunks)} real chunks from the Helios corpus")

chunk_vectors = embed(chunks)                       # small batch — safe and fast
query_vector = embed(["What payload can the HeliosArm V2 carry continuously?"])
sims = cosine_similarity(query_vector, chunk_vectors)
for idx in sims.argsort()[::-1][:3]:
    print(f"  score={sims[idx]:.3f}  {chunks[idx][:90]}")


## The scaling problem, measured

To see *why* you'd ever reach for an index, we need many more vectors than the Helios corpus has. Embedding millions of real documents would take ages (and stress your machine), so we use **synthetic random unit vectors** of the same dimension — for a *timing* comparison, only the vector count and dimension matter, not their meaning.


In [ ]:
dim = chunk_vectors.shape[1]   # 384, same as the real embeddings
rng = np.random.default_rng(0)

def random_unit_vectors(n, d):
    v = rng.standard_normal((n, d)).astype(np.float32)
    v /= (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)
    return v

def brute_force_search(query, vectors, k=5):
    sims = vectors @ query                # vectors are unit-norm, so dot == cosine
    return sims.argsort()[::-1][:k]

big = random_unit_vectors(50_000, dim)    # 50k vectors — still tiny for a real DB
q = random_unit_vectors(1, dim)[0]

t0 = time.perf_counter(); brute_idx = brute_force_search(q, big, k=5); t_brute = time.perf_counter() - t0

index = faiss.IndexHNSWFlat(dim, 32)      # HNSW: a graph-based ANN index
index.add(big)
t0 = time.perf_counter(); _, ann_idx = index.search(q.reshape(1, -1), 5); t_ann = time.perf_counter() - t0

print(f"Brute-force over 50k: {t_brute*1000:7.2f} ms")
print(f"FAISS HNSW over 50k:  {t_ann*1000:7.2f} ms")
print(f"Top-5 overlap (recall proxy): {len(set(brute_idx.tolist()) & set(ann_idx[0].tolist()))}/5")


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

sizes = [1_000, 10_000, 50_000, 100_000]
brute_times, ann_times = [], []
for n in sizes:
    vecs = random_unit_vectors(n, dim)
    t0 = time.perf_counter(); brute_force_search(q, vecs, k=5); brute_times.append(time.perf_counter() - t0)
    idx_n = faiss.IndexHNSWFlat(dim, 32); idx_n.add(vecs)
    t0 = time.perf_counter(); idx_n.search(q.reshape(1, -1), 5); ann_times.append(time.perf_counter() - t0)

plt.figure(figsize=(7, 4))
plt.plot(sizes, [t*1000 for t in brute_times], marker="o", label="Brute-force cosine")
plt.plot(sizes, [t*1000 for t in ann_times], marker="o", label="FAISS HNSW (ANN)")
plt.xlabel("Number of vectors"); plt.ylabel("Search time (ms)")
plt.title("Brute-force vs. ANN search time as corpus grows")
plt.legend(); plt.tight_layout(); plt.show()


*Brute-force search time grows roughly linearly with corpus size since every vector is compared; the ANN index's graph traversal grows far more slowly, which is the entire value proposition at production scale.*


## The same thing, via `ragkit.vectorstore` (Chroma)

In practice you won't hand-build a FAISS index per notebook — `ragkit.vectorstore` already wraps Chroma (a vector database, not just an index) with persistence and metadata filtering, and it's what every notebook in `07_rag_learning/` uses.


In [ ]:
from ragkit.vectorstore import build_collection, query_collection
from ragkit.data import build_chunked_corpus

texts, metadatas = build_chunked_corpus(chunk_size=120, overlap=20)
collection = build_collection("helios_tier3_demo", texts, metadatas, persist_dir="../.chroma", reset=True)

hits = query_collection(collection, "What payload can the HeliosArm V2 carry continuously?", k=3)
for hit in hits:
    print(f"score={hit.score:.3f}  source={hit.metadata.get('source')}  text={hit.text[:80]}")


## Where this notebook ends

Pinecone (or another managed vector DB) is the same ANN idea hosted as a service, useful once your index outgrows a single machine or needs multi-region availability. It's optional here — set `PINECONE_API_KEY`/`PINECONE_ENV` in `.env` to try the cell below; otherwise it's skipped, and everything you learned with FAISS/Chroma transfers directly.


In [ ]:
HAS_PINECONE = bool(os.environ.get("PINECONE_API_KEY"))

if HAS_PINECONE:
    try:
        from pinecone import Pinecone
        pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
        print("Connected to Pinecone. Existing indexes:", [i["name"] for i in pc.list_indexes()])
    except Exception as e:
        print(f"Pinecone call failed (package not installed or other issue): {e}")
else:
    print("No PINECONE_API_KEY set — skipping. FAISS/Chroma above already demonstrate the same ANN concept.")


## Exercises


In [ ]:
# Exercise 1 (Warm-up): Change the ANN parameter
# Task: Rebuild the FAISS index with index = faiss.IndexHNSWFlat(dim, 4) (a much smaller graph
#       degree than 32) over `big` and re-run the search. Does overlap with brute-force drop?
# Hint: Smaller graph degree means fewer connections to explore -> faster but less accurate;
#       this is the recall/speed knob every ANN index exposes.

# YOUR CODE HERE


In [ ]:
# Exercise 2 (Apply): Build an IVF index instead of HNSW
# Task: Build a faiss.IndexIVFFlat (cluster-based ANN) over `big`, train it, and compare its
#       search time and top-5 overlap against both brute-force and HNSW.
# Hint: IVF needs `index.train(vectors)` before `.add()` — unlike HNSW, which builds its graph
#       incrementally as you add vectors.

# YOUR CODE HERE


In [ ]:
# Exercise 3 (Extend): Metadata-filtered retrieval
# Task: Query the Chroma `collection` for only chunks where metadata category == "incident",
#       using Chroma's where-filter. Compare to an unfiltered query for the same question.
# Hint: collection.query(query_texts=[...], n_results=3, where={"category": "incident"}) —
#       this is something a from-scratch FAISS index does NOT give you without extra bookkeeping.

# YOUR CODE HERE


<details>
<summary>Show solutions</summary>

```python
# Exercise 1
small_index = faiss.IndexHNSWFlat(dim, 4)
small_index.add(big)
_, small_ann = small_index.search(q.reshape(1, -1), 5)
print("Overlap with brute-force:", len(set(brute_idx.tolist()) & set(small_ann[0].tolist())), "/5")
# A smaller graph degree usually reduces recall (fewer neighbors explored per node).

# Exercise 2
nlist = 50
quantizer = faiss.IndexFlatIP(dim)
ivf = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
ivf.train(big); ivf.add(big)
t0 = time.perf_counter(); _, ivf_idx = ivf.search(q.reshape(1, -1), 5)
print(f"IVF search: {(time.perf_counter()-t0)*1000:.2f} ms, overlap with brute-force:",
      len(set(brute_idx.tolist()) & set(ivf_idx[0].tolist())), "/5")

# Exercise 3
filtered = collection.query(query_texts=["incident affecting the gripper"], n_results=3,
                            where={"category": "incident"})
for doc, meta in zip(filtered["documents"][0], filtered["metadatas"][0]):
    print(meta.get("source"), "->", doc[:80])
```
</details>


## Key Takeaways
- Brute-force cosine search compares the query to every vector; it's exact but scales linearly with corpus size.
- An ANN index (HNSW, IVF, ...) trades a small amount of recall for retrieval that scales far better than linear.
- Vector databases (Chroma, Pinecone, ...) wrap an ANN index with persistence, metadata filtering, and incremental updates — that's the production-ready layer, not just the index.
- `ragkit.vectorstore` already wraps Chroma; every notebook in `07_rag_learning/` builds on it.
- Pinecone is the same idea as a managed service — useful once a single machine's index isn't enough.

## What's Next
Notebook **16 — RAG vs. CAG** revisits notebook 13's prompt caching and contrasts it directly against retrieval: when is it cheaper to *cache* knowledge in the prompt versus *retrieve* it fresh from the vector store you just built?
